In [ ]:
!pip install tsfel -q

import tsfel
import numpy as np
import pandas as pd
from tqdm import tqdm

#FEATURE EXTRACTION

In [ ]:
META_COLS = [
    "segment_id", "subject_id", "dataset", "condition",
    "start_time", "end_time", "n_rr", "label",
]

In [ ]:
# FITUR HRV KLASIKK
def compute_time_domain(rr_ms):
    rr = np.asarray(rr_ms, dtype=float)
    sdnn = np.std(rr, ddof=1)

    if sdnn > 300:
        print("="*50)
        print("SDNN BESAR")
        print(f"SDNN : {sdnn:.2f}")
        print(f"Mean : {np.mean(rr):.2f}")
        print(f"Min  : {np.min(rr):.2f}")
        print(f"Max  : {np.max(rr):.2f}")
        print("20 RR pertama:")
        print(rr[:20])
        print("="*50)

    if len(rr) < 2:
        return {k: np.nan for k in ["SDNN","RMSSD","pNN50","median_rr","cv_rr"]}
    diff = np.diff(rr)
    return {
        "SDNN"     : round(float(np.std(rr, ddof=1)),                        4),
        "RMSSD"    : round(float(np.sqrt(np.mean(diff**2))),                 4),
        "pNN50"    : round(float(np.sum(np.abs(diff) > 50) / len(diff)*100), 4),
        "median_rr": round(float(np.median(rr)),                             4),
        "cv_rr"    : round(float(np.std(rr) / (np.mean(rr) + 1e-8)),        6),
    }

#FITUR TSFEL temporal
def extract_features(all_segments, fs=4.0):
    cfg  = tsfel.get_features_by_domain("temporal")
    rows = []

    for i, seg in enumerate(tqdm(all_segments, desc="feature extraction")):
        rr_ms = np.asarray(seg["rr_ms"], dtype=float)
        if len(rr_ms) < 10:
            continue

        manual = compute_time_domain(rr_ms)

        try:
            tsfel_feats = tsfel.time_series_features_extractor(
                cfg, pd.Series(rr_ms), fs=fs, verbose=0
            )
            tsfel_dict = tsfel_feats.iloc[0].to_dict()
        except Exception as e:
            print(f"  [TSFEL] skip segmen {i}: {e}")
            tsfel_dict = {}

        rows.append({
            "segment_id": i,
            "subject_id": seg.get("subject_id", ""),
            "dataset"   : seg.get("dataset", ""),
            "condition" : seg.get("condition", ""),
            "start_time": seg.get("start_time", 0),
            "end_time"  : seg.get("end_time", 0),
            "n_rr"      : len(rr_ms),
            "label"     : seg.get("label", np.nan),

            **manual,
            **tsfel_dict,
        })

    df = pd.DataFrame(rows)
    print(f"\n {len(df)} segmen | {len(df.columns)} kolom")

    for lbl in ["normal", "stress_berkendara", "stress_akademik"]:
        print(f"   {lbl:20s}: {(df['label'] == lbl).sum()}")
    return df

In [ ]:
df_features = extract_features(all_segments, fs=4.0)

output_csv = os.path.join(data_path, "ekstraksi_fitur_revisi.csv")
df_features.to_csv(output_csv, index=False)
print(f"Tersimpan: {output_csv}")

#FEATURE ENGINEERING
df_features = df_features.copy()

for feat in ["SDNN", "RMSSD", "pNN50", "cv_rr"]:
    df_features[f"log_{feat}"] = np.log(df_features[feat].clip(lower=1e-8))

df_features["RMSSD_SDNN_ratio"] = df_features["RMSSD"] / (df_features["SDNN"] + 1e-8)
df_features["pNN50_norm"]       = df_features["pNN50"] / (df_features["median_rr"] + 1e-8)

for feat in ["SDNN", "RMSSD", "pNN50"]:
    subj_mean = df_features.groupby("subject_id")[feat].transform("mean")
    df_features[f"{feat}_delta"] = df_features[feat] - subj_mean

feature_cols = [c for c in df_features.columns if c not in META_COLS]
print(f"Total fitur setelah: {len(feature_cols)}")
print(df_features[["segment_id", "subject_id", "dataset", "label",
                    "SDNN", "RMSSD", "pNN50", "median_rr", "cv_rr"]].head(8).to_string(index=False))

DATASET VALIDATION

In [ ]:
try:
    df_features
    print(f"df_features sudah ada: {df_features.shape}")
except NameError:
    output_csv  = os.path.join(data_path, "ekstraksi_fitur.csv")
    df_features = pd.read_csv(output_csv)
    print(f"Loaded dari CSV: {df_features.shape}")

fitur_hrv = ["SDNN", "RMSSD", "pNN50"]
LABELS    = ["normal", "stress_berkendara", "stress_akademik"]
LABEL_COLORS = {
    "normal"            : "#6BCB77",
    "stress_berkendara" : "#FFB84D",
    "stress_akademik"   : "#FF6B6B",
}

# DISTRIBUSI LABEL
print("\n" + "=" * 60)
print("1. DISTRIBUSI LABEL")
print("=" * 60)

total  = len(df_features)
counts = {lbl: (df_features["label"] == lbl).sum() for lbl in LABELS}

for lbl in LABELS:
    print(f"  {lbl:20s}: {counts[lbl]} segmen ({counts[lbl]/total*100:.1f}%)")
print(f"  Total               : {total} segmen")
print(f"  Ratio majority:minority = {max(counts.values())/min(counts.values()):.2f}x")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

label_counts = df_features["label"].value_counts().reindex(LABELS)
axes[0].bar(label_counts.index, label_counts.values,
            color=[LABEL_COLORS[l] for l in label_counts.index],
            edgecolor="white", linewidth=1.5)
axes[0].set_title("Distribusi Label", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Jumlah Segmen")
axes[0].tick_params(axis="x", rotation=15)
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 5, str(v), ha="center", fontweight="bold")

dataset_label = df_features.groupby(["dataset", "label"]).size().unstack(fill_value=0)
dataset_label = dataset_label.reindex(columns=LABELS, fill_value=0)
dataset_label.plot(kind="bar", ax=axes[1],
                   color=[LABEL_COLORS[l] for l in dataset_label.columns],
                   edgecolor="white")
axes[1].set_title("Distribusi Label per Dataset", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Dataset")
axes[1].set_ylabel("Jumlah Segmen")
axes[1].tick_params(axis="x", rotation=15)
axes[1].legend(title="Label")

plt.tight_layout()
plt.savefig(os.path.join(data_path, "validasi_distribusi_label.png"), dpi=150)
plt.show()
print("Plot tersimpan: validasi_distribusi_label.png")

#STATISTIK DESKRIPTIF PER DATASET
print("\n" + "=" * 60)
print("STATISTIK DESKRIPTIF PER DATASET")
print("=" * 60)

stat_dataset = df_features.groupby("dataset")[fitur_hrv].agg(["mean", "std", "min", "max"])
stat_dataset.columns = [f"{col}_{stat}" for col, stat in stat_dataset.columns]
print("\nPer Dataset:")
print(stat_dataset.round(2).to_string())

print("\nPer Label (mean ± std):")
stat_label = df_features.groupby("label")[fitur_hrv].agg(
    lambda x: f"{x.mean():.2f} ± {x.std():.2f}"
).T
stat_label.index = ["SDNN (ms)", "RMSSD (ms)", "pNN50 (%)"]
print(stat_label.to_string())

# STATISTIK PER LABEL PER DATASET
print("\n" + "=" * 60)
print("STATISTIK PER LABEL PER DATASET")
print("=" * 60)

stat_label_dataset = df_features.groupby(["dataset", "label"])[fitur_hrv].agg(
    lambda x: f"{x.mean():.2f} ± {x.std():.2f}"
).reset_index()
print(stat_label_dataset.to_string(index=False))

# CORRELATION MATRIX FITUR HRV
print("\n" + "=" * 60)
print("CORRELATION MATRIX FITUR HRV")
print("=" * 60)

corr_matrix = df_features[fitur_hrv].corr()
print(corr_matrix.round(3).to_string())

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(fitur_hrv)))
ax.set_yticks(range(len(fitur_hrv)))
ax.set_xticklabels(fitur_hrv, fontsize=11)
ax.set_yticklabels(fitur_hrv, fontsize=11)

for i in range(len(fitur_hrv)):
    for j in range(len(fitur_hrv)):
        ax.text(j, i, f"{corr_matrix.iloc[i,j]:.2f}",
                ha="center", va="center", fontsize=12, fontweight="bold")

ax.set_title("Correlation Matrix Fitur HRV", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(data_path, "validasi_correlation_matrix.png"), dpi=150)
plt.show()
print("Plot tersimpan: validasi_correlation_matrix.png")

#DISTRIBUSI NILAI FITUR HRV
print("\n" + "=" * 60)
print("DISTRIBUSI NILAI FITUR HRV")
print("=" * 60)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for col_idx, fitur in enumerate(fitur_hrv):
    ax = axes[0, col_idx]
    for label, group in df_features.groupby("label"):
        ax.hist(group[fitur].dropna(), bins=30, alpha=0.6,
                label=label, color=LABEL_COLORS[label], edgecolor="white")
    ax.set_title(f"Distribusi {fitur}", fontsize=12, fontweight="bold")
    ax.set_xlabel(f"{fitur} (ms)" if fitur != "pNN50" else "pNN50 (%)")
    ax.set_ylabel("Frekuensi")
    ax.legend()

    ax = axes[1, col_idx]
    df_features.boxplot(column=fitur, by="dataset", ax=ax,
                        patch_artist=True,
                        boxprops=dict(facecolor="#AED9E0"),
                        medianprops=dict(color="red", linewidth=2))
    ax.set_title(f"Boxplot {fitur} per Dataset", fontsize=12, fontweight="bold")
    ax.set_xlabel("Dataset")
    ax.set_ylabel(f"{fitur} (ms)" if fitur != "pNN50" else "pNN50 (%)")
    plt.sca(ax)
    plt.xticks(rotation=15)

plt.suptitle("Distribusi Fitur HRV", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(data_path, "validasi_distribusi_fitur.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Plot tersimpan: validasi_distribusi_fitur.png")

#MISSING VALUES
print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing_all = df_features.isnull().sum()
missing_all = missing_all[missing_all > 0]

if len(missing_all) == 0:
    print("Tidak ada missing values di seluruh dataset!")
else:
    print(f"Ditemukan {len(missing_all)} kolom dengan missing values:\n")
    missing_df = pd.DataFrame({
        "Kolom"      : missing_all.index,
        "Jumlah NaN" : missing_all.values,
        "Persentase" : (missing_all.values / len(df_features) * 100).round(2)
    })
    print(missing_df.to_string(index=False))

    cols_drop = missing_all[missing_all > len(df_features) * 0.5].index.tolist()
    if cols_drop:
        print(f"Drop {len(cols_drop)} kolom (NaN > 50%): {cols_drop}")
        df_features = df_features.drop(columns=cols_drop)
        print("Selesai di-drop")

print("\nCek missing values fitur HRV utama:")
for fitur in fitur_hrv:
    n_missing = df_features[fitur].isnull().sum()
    status    = "OK" if n_missing == 0 else f"⚠️ {n_missing} missing"
    print(f"  {fitur:10s}: {status}")

# RINGKASAN VALIDASI
print("\n" + "=" * 60)
print("RINGKASAN VALIDASI KUALITAS DATASET")
print("=" * 60)
print(f"  Total segmen         : {len(df_features)}")
print(f"  Total fitur          : {len(feature_cols)}")
for lbl in LABELS:
    c = counts[lbl]
    print(f"  {lbl:20s} : {c} ({c/total*100:.1f}%)")
print(f"  Missing values       : {'Tidak ada' if len(missing_all)==0 else f'{len(missing_all)} kolom'}")
print(f"  Dataset sumber       : {df_features['dataset'].unique().tolist()}")